# Cogni Chunk ??: Intelligent Technical Document Analyst
## Personal Project Notebook Walkthrough ?

This notebook presents a polished, self-contained retrieval demo for technical documentation.
Instead of relying on external vector databases or hosted models, it uses a local structure-aware retrieval pipeline so the project stays reliable, portable, and easy to explain.


## 1. Demo Goals

We want the project to communicate four things clearly:

- Technical documents need better retrieval than plain keyword search.
- Chunking by structure and section meaning improves retrieval quality.
- Answers should be grounded in visible evidence.
- The same retrieval engine should power both the notebook and the Streamlit UI.


In [17]:
from pathlib import Path
from cogni_chunk_engine import TechnicalDocAnalyst, run_cli_query

DOC_PATH = Path('technical_doc.md')
analyst = TechnicalDocAnalyst(DOC_PATH)

print(f'Document: {DOC_PATH.resolve()}')
print(f'Sections indexed: {len(analyst.chunks)}')


Document: D:\aiLearning\CogniChunk\technical_doc.md
Sections indexed: 24


## 2. Inspect The Source Document 

The corpus is intentionally richer now. It includes architecture decisions, performance benchmarks, failure modes, security notes, observability metrics, and incident case studies so retrieval feels realistic.


In [18]:
doc_preview = DOC_PATH.read_text(encoding='utf-8')
print(doc_preview[:2500] + '...')


# Technical Design Dossier: Atlas Knowledge Fabric

## 1. Executive Overview
Atlas Knowledge Fabric is a distributed retrieval and reasoning platform designed for enterprise technical documentation, change logs, runbooks, architecture decision records, and post-incident analysis. The platform serves engineering, support, and operations teams that need fast answers from large internal corpora without losing source attribution. The core product goal is to reduce the time between a user question and a trustworthy technical answer.

The design prioritizes four properties:
1. Precision under ambiguity.
2. Low-latency retrieval at scale.
3. Explainable answers with visible evidence.
4. Operational resilience during partial infrastructure failure.

Atlas is typically deployed across three logical planes: an ingestion plane, an indexing plane, and a query execution plane. Each plane scales independently and can be throttled without fully disabling the others. This isolation reduces blast radiu

## 3. Structure-Aware Chunking 

Each markdown heading becomes a natural retrieval section, and the heading path is preserved as metadata. This makes the project easier to reason about than arbitrary character windows.


In [19]:
for index, chunk in enumerate(analyst.chunks[:6], start=1):
    print(f'--- Chunk {index} ---')
    print('Heading path:', chunk.heading_path)
    print('Token count:', chunk.token_count)
    print(chunk.content[:350] + ('...' if len(chunk.content) > 350 else ''))
    print()


--- Chunk 1 ---
Heading path: Technical Design Dossier: Atlas Knowledge Fabric > 1. Executive Overview
Token count: 106
Atlas Knowledge Fabric is a distributed retrieval and reasoning platform designed for enterprise technical documentation, change logs, runbooks, architecture decision records, and post-incident analysis. The platform serves engineering, support, and operations teams that need fast answers from large internal corpora without losing source attributio...

--- Chunk 2 ---
Heading path: Technical Design Dossier: Atlas Knowledge Fabric > 2. Problem Statement
Token count: 72
Traditional enterprise search often fails for technical documents because terminology is inconsistent and critical knowledge is buried inside long paragraphs, code samples, and operational caveats. Simple keyword search does not understand that "replication lag", "write delay", and "commit propagation latency" may refer to overlapping phenomena. Na...

--- Chunk 3 ---
Heading path: Technical Design Dossi

## 4. Retrieval Logic 

The scoring function blends term overlap, inverse document frequency, heading-path matches, and a small boost for evidence-rich language such as causal explanations or latency figures.


In [20]:
query = 'Why did retrieval latency improve by about 40 percent in release AKF-2.3?'
results = analyst.search(query, top_k=4)

for result in results:
    print(result.chunk.heading_path)
    print('score =', round(result.score, 3))
    print('matched =', ', '.join(result.matched_terms))
    print()


Technical Design Dossier: Atlas Knowledge Fabric > 11. Performance Engineering
score = 1.679
matched = 3, 40, akf-2, latency, percent, release, retrieval

Technical Design Dossier: Atlas Knowledge Fabric > 19. Sample Questions the System Should Answer Well
score = 1.183
matched = 3, 40, about, akf-2, did, improve, latency, percent, release, retrieval

Technical Design Dossier: Atlas Knowledge Fabric > 16. Case Study: Reindex Storm After Documentation Migration
score = 0.727
matched = akf-2, latency, release

Technical Design Dossier: Atlas Knowledge Fabric > 15. Case Study: Replica Failover Incident
score = 0.591
matched = 40, latency, percent



## 5. Example Grounded Queries

These sample prompts are designed to show breadth: performance, failure handling, operational incidents, and architecture reasoning.


In [21]:
demo_queries = [
    'What happens when the vector index is unavailable?',
    'Why were write-heavy dashboards slower during the replica failover test?',
    'How does Atlas handle ingestion storms during large document migrations?',
    'What metrics should on-call engineers watch after a retriever deployment?',
]

for q in demo_queries:
    print(run_cli_query(q))
    print('' + '=' * 100 + '')


Document: technical_doc.md
Query: What happens when the vector index is unavailable?
Confidence: high

The strongest evidence points to '12.1 Vector Index Degradation'. If the vector index becomes unavailable, Atlas falls back to lexical retrieval only. This answer is grounded in sections about 12.1 Vector Index Degradation.

Top Evidence:
1. Technical Design Dossier: Atlas Knowledge Fabric > 12. Failure Modes and Recovery > 12.1 Vector Index Degradation | score=1.851 | matched=index, unavailable, vector
   If the vector index becomes unavailable, Atlas falls back to lexical retrieval only. Recall drops on conceptual questions, but identifier-based lookups still work. The UI surfaces a reduced-confidence banner in this mode....
2. Technical Design Dossier: Atlas Knowledge Fabric > 3. Reference Deployment Topology | score=0.645 | matched=index, vector
   The reference deployment uses the following components: - Edge API Gateway for authentication, rate limiting, and request shaping. - Q

## 6. Optional UI Demo 

```bash
streamlit run app.py
```
